# Content-Based Movie Recommendation System using NLP

In [ ]:
pip install pandas numpy scikit-learn nltk

## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import ast

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Loading Dataset

In [ ]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

## Data Preprocessing

In [ ]:
movies.head()

In [ ]:
credits.head()

In [ ]:
movies.columns

In [ ]:
movies = movies.merge(credits, on='title')

In [ ]:
movies = movies[['movie_id',
                 'title',
                 'overview',
                 'genres',
                 'keywords',
                 'cast',
                 'crew']]

In [ ]:
movies.head()

In [ ]:
movies.isnull().sum()

In [ ]:
movies.dropna(inplace=True)

## Feature Engineering

In [ ]:
'[{"id": 28, "name": "Action"}]'

In [ ]:
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

In [ ]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
def convert_cast(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

In [ ]:
movies['cast'] = movies['cast'].apply(convert_cast)

In [ ]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [ ]:
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

In [ ]:
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
new = movies[['movie_id','title','tags']].copy()

In [ ]:
new['tags'] = new['tags'].apply(lambda x: " ".join(x))

In [ ]:
new['tags'] = new['tags'].apply(lambda x:x.lower())

## Vectorization

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words='english')

In [ ]:
vectors = cv.fit_transform(new['tags']).toarray()

## Cosine Similarity

In [ ]:
similarity = cosine_similarity(vectors)

## Recommendation Function

In [ ]:
def recommend(movie):

    if movie not in new['title'].values:
        print("Movie not found in dataset.")
        return

    movie_index = new[new['title'] == movie].index[0]

    distances = similarity[movie_index]

    movies_list = sorted(list(enumerate(distances)),
                         reverse=True,
                         key=lambda x:x[1])[1:6]

    print("Recommended Movies:\n")

    for i in movies_list:
        print(new.iloc[i[0]].title)

## Testing Recommendation System

In [ ]:
recommend('Avatar')

In [ ]:
recommend('Batman Begins')

## Data Visualization

# CHART 1 — Top 10 Movie Genres
This chart shows:
(a) which genres appear most in dataset
(b) distribution of movie categories

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

genre_list = []

for i in movies['genres']:
    genre_list.extend(i)

counter = Counter(genre_list)

top_genres = counter.most_common(10)

x = [i[0] for i in top_genres]
y = [i[1] for i in top_genres]

plt.figure(figsize=(10,5))
plt.bar(x, y)

plt.xlabel("Genres")
plt.ylabel("Count")
plt.title("Top 10 Movie Genres")

plt.xticks(rotation=45)

plt.show()

# CHART 2 — Most Frequent Actors

In [ ]:
actor_list = []

for i in movies['cast']:
    actor_list.extend(i)

counter = Counter(actor_list)

top_actors = counter.most_common(10)

x = [i[0] for i in top_actors]
y = [i[1] for i in top_actors]

plt.figure(figsize=(12,5))
plt.bar(x, y)

plt.xlabel("Actors")
plt.ylabel("Movie Count")
plt.title("Top 10 Most Frequent Actors")

plt.xticks(rotation=45)

plt.show()

# CHART 3 — Most Common Keywords

In [ ]:
keyword_list = []

for i in movies['keywords']:
    keyword_list.extend(i)

counter = Counter(keyword_list)

top_keywords = counter.most_common(10)

x = [i[0] for i in top_keywords]
y = [i[1] for i in top_keywords]

plt.figure(figsize=(12,5))
plt.bar(x, y)

plt.xlabel("Keywords")
plt.ylabel("Frequency")
plt.title("Top 10 Most Common Keywords")

plt.xticks(rotation=45)

plt.show()

### Similarity Heatmap

In [ ]:
import seaborn as sns

plt.figure(figsize=(8,6))

sns.heatmap(similarity[:20,:20])

plt.title("Movie Similarity Heatmap")

plt.show()

## Technologies Used

- Python
- Pandas
- NumPy
- Scikit-learn
- NLP
- Cosine Similarity
- Matplotlib

## Conclusion

This project successfully implements a content-based movie recommendation system using NLP techniques and cosine similarity. The system recommends movies based on similarity of genres, cast, keywords, and overview information.